# Task 1 — Clean Scikit-Learn Sales Prediction

Same workflow as before, but **without Pipeline**. This version keeps the steps manual and easier to follow.

In [ ]:
# =========================
# Sales Prediction - Task 1
# Scikit-Learn Workflow WITHOUT Pipeline
# =========================

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from pathlib import Path
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


# -------------------------
# 1. Load the dataset
# -------------------------

# Put sales.csv in the same folder as this notebook.
DATA_PATH = Path("sales.csv")

df_sales = pd.read_csv(DATA_PATH)

print("Dataset shape:", df_sales.shape)
display(df_sales.head())


# -------------------------
# 2. Quick data check
# -------------------------

print("\nDataset info:")
df_sales.info()

print("\nMissing values:")
display(df_sales.isna().sum())

print("\nSummary statistics:")
display(df_sales.describe(include="all").T)


# -------------------------
# 3. Basic cleaning
# -------------------------

# Remove accidental index column if it exists
df_sales = df_sales.drop(columns=["Unnamed: 0"], errors="ignore")

# Convert date into useful numeric features, then remove original date column
if "date" in df_sales.columns:
    df_sales["date"] = pd.to_datetime(df_sales["date"], errors="coerce")
    df_sales["year"] = df_sales["date"].dt.year
    df_sales["month"] = df_sales["date"].dt.month
    df_sales["day"] = df_sales["date"].dt.day
    df_sales["day_of_week"] = df_sales["date"].dt.dayofweek
    df_sales = df_sales.drop(columns=["date"])

# Remove rows where the target is missing
df_sales = df_sales.dropna(subset=["sales"])

print("\nCleaned dataset shape:", df_sales.shape)
display(df_sales.head())


# -------------------------
# 4. Split features and target
# -------------------------

X = df_sales.drop(columns=["sales"])
y = df_sales["sales"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)

print("Training rows:", X_train.shape[0])
print("Testing rows:", X_test.shape[0])


# -------------------------
# 5. Encode categorical columns manually
# -------------------------

# Turn text/categorical columns into numeric dummy columns
X_train = pd.get_dummies(X_train, drop_first=True)
X_test = pd.get_dummies(X_test, drop_first=True)

# Make sure train and test have the exact same columns
X_train, X_test = X_train.align(X_test, join="left", axis=1, fill_value=0)

# Fill any remaining missing values with 0
X_train = X_train.fillna(0)
X_test = X_test.fillna(0)

print("\nFinal training columns:", X_train.shape[1])
display(X_train.head())


# -------------------------
# 6. Train the model
# -------------------------

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained successfully.")


# -------------------------
# 7. Make predictions
# -------------------------

y_pred = model.predict(X_test)


# -------------------------
# 8. Evaluate the model
# -------------------------

mae = mean_absolute_error(y_test, y_pred)
rmse = np.sqrt(mean_squared_error(y_test, y_pred))
r2 = r2_score(y_test, y_pred)

print(f"MAE:  {mae:.2f}")
print(f"RMSE: {rmse:.2f}")
print(f"R²:   {r2:.4f}")


# -------------------------
# 9. Plot actual vs predicted
# -------------------------

plt.figure(figsize=(8, 6))
plt.scatter(y_test, y_pred, alpha=0.5)
plt.xlabel("Actual Sales")
plt.ylabel("Predicted Sales")
plt.title("Actual vs Predicted Sales")
plt.show()
